# 04. scikit-learn 텍스트 분류 기준 모델

복잡한 신경망 전에 빠르고 설명 가능한 기준선을 만듭니다.
이 노트북은 **정규화 → TF-IDF/One-Hot → ColumnTransformer → Pipeline → Logistic Regression →
metrics/confusion matrix/error analysis**를 연결하고, group leakage가 평가를 어떻게 왜곡하는지 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=sys.path.insert -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """현재 폴더부터 위로 올라가며 프로젝트 루트를 찾습니다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "pyproject.toml과 src/가 있는 프로젝트 폴더를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("project:", PROJECT_ROOT)
print("python :", sys.executable)

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.train -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from llm_engineering_lab.data import (
    TicketSchema,
    load_ticket_data,
    stratified_group_split,
)
from llm_engineering_lab.ml import (
    TrainingConfig,
    build_error_table,
    build_model,
    evaluate_model,
    normalize_ticket_text,
    predict_tickets,
)

tickets = load_ticket_data(PROJECT_ROOT / "data" / "customer_support_tickets.csv")
schema = TicketSchema()
split = stratified_group_split(tickets, validation_size=0.2, seed=42)
print("train:", split.train.shape, "validation:", split.validation.shape)

## 1. 텍스트 정규화

Unicode 표현과 공백을 통일하고 이메일·긴 숫자처럼 샘플마다 달라지는 식별자를 마스킹합니다.
레이블 정보를 암시하는 후처리나 전체 데이터 통계를 사용하면 안 됩니다.

## 2. TF-IDF를 작은 문장으로 해부하기

TF-IDF는 한 문서 안에서 자주 나오지만 전체 문서에는 흔하지 않은 항목을 강조합니다.
한국어 형태소 분석기 없이도 문자 n-gram은 오탈자와 조사 변화에 비교적 강한 기준선이 됩니다.

In [ ]:
normalization_examples = [
    "  카드 결제가   두 번 됐어요  ",
    "User.Test@example.com 계정의 주문 번호는 123456 입니다",
    "ＡＰＩ 오류가 발생했습니다",  # 전각 문자
]

normalized = [normalize_ticket_text(text) for text in normalization_examples]
assert "<email>" in normalized[1]
assert "<number>" in normalized[1]
normalized

In [ ]:
tiny_corpus = [
    "카드 결제가 두 번 됐어요",
    "카드 결제를 취소해 주세요",
    "배송이 너무 늦어요",
]
tiny_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
tiny_matrix = tiny_vectorizer.fit_transform(tiny_corpus)

print("sparse shape:", tiny_matrix.shape)
print("non-zero values:", tiny_matrix.nnz)
assert tiny_matrix.shape[0] == 3

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.argsort,pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.argsort(a, axis=-1, kind=None)</code></summary>

#### `np.argsort(a, axis=-1, kind=None)`

- **역할:** 정렬 결과를 만드는 index 순서를 반환합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

</details>

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
feature_names = tiny_vectorizer.get_feature_names_out()
first_row = tiny_matrix[0].toarray().ravel()
top_indices = np.argsort(first_row)[-8:][::-1]
pd.DataFrame(
    {
        "ngram": feature_names[top_indices],
        "tfidf": first_row[top_indices],
    }
)

## 3. 범주형 metadata와 unknown 처리

`OneHotEncoder(handle_unknown="ignore")`는 추론 시 처음 보는 channel이 들어와도 전체 요청을 실패시키지 않습니다.
unknown이 모두 0으로 표현된다는 운영 의미는 별도로 모니터링해야 합니다.

## 4. `Pipeline`으로 전처리와 모델을 묶기

`fit`은 train에만 호출합니다. validation에 `fit_transform`을 다시 호출하면 vocabulary와 category 공간이 달라지고 누수가 생깁니다.
Pipeline은 저장·추론 때도 동일한 변환을 재사용하게 해 줍니다.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_channels = pd.DataFrame({"channel": ["web", "email", "chat"]})
encoded_train = encoder.fit_transform(train_channels)
encoded_new = encoder.transform(pd.DataFrame({"channel": ["phone"]}))

assert encoded_train.shape == (3, 3)
assert np.all(encoded_new == 0)
pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out())

In [ ]:
config = TrainingConfig(
    seed=42,
    validation_size=0.2,
    max_features=20_000,
    ngram_range=(2, 5),
    regularization_c=4.0,
)
model = build_model(config, schema)
print(model)

<!-- api-explanations:start -->
<!-- api-explanations:keys=Estimator.fit,Estimator.predict -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>estimator.fit(X, y=None, **fit_params)</code></summary>

#### `estimator.fit(X, y=None, **fit_params)`

- **역할:** 데이터에서 estimator의 파라미터나 통계를 학습합니다.
- **입력·반환:** feature `X`와 선택적 target `y`를 받고 보통 estimator 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 split만 사용해 모델·전처리기를 적합합니다.
- **주의점:** test 데이터에 `fit`하면 데이터 누수가 생기며 estimator별 입력 계약이 다릅니다.
- **공식 문서:** [Estimator.fit](https://scikit-learn.org/stable/glossary.html#term-fit)

</details>

<details>
<summary><code>estimator.predict(X)</code></summary>

#### `estimator.predict(X)`

- **역할:** 학습된 estimator로 sample별 예측을 만듭니다.
- **입력·반환:** feature 행렬을 받고 sample 축 길이의 예측 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 검증·test split에서 최종 label이나 값을 평가합니다.
- **주의점:** 먼저 `fit`되어야 하며 분류 확률이 필요한 경우와 구분해야 합니다.
- **공식 문서:** [Estimator.predict](https://scikit-learn.org/stable/glossary.html#term-predict)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_x = split.train.loc[:, schema.model_feature_columns]
train_y = split.train[schema.label]
validation_x = split.validation.loc[:, schema.model_feature_columns]
validation_y = split.validation[schema.label]

model.fit(train_x, train_y)
predictions = model.predict(validation_x)

assert len(predictions) == len(validation_y)
assert set(predictions).issubset(set(train_y))
print("학습 완료")

## 5. Pipeline 내부 shape 확인

전처리 결과는 대부분 0인 sparse matrix입니다. 대규모 vocabulary를 dense 배열로 바꾸면 메모리가 급증합니다.

## 6. 확률과 예측

가장 높은 확률은 불확실성 순위를 정하는 신호이지, 자동으로 보정된 확률을 뜻하지는 않습니다.
클래스 순서는 `classifier.classes_`와 함께 확인합니다.

In [ ]:
preprocessor = model.named_steps["preprocessing"]
transformed_train = preprocessor.transform(train_x)

print("type :", type(transformed_train).__name__)
print("shape:", transformed_train.shape)
print("nnz  :", transformed_train.nnz)
density = transformed_train.nnz / (
    transformed_train.shape[0] * transformed_train.shape[1]
)
print("density:", round(density, 6))
assert density < 0.5

<!-- api-explanations:start -->
<!-- api-explanations:keys=Estimator.predict_proba,np.allclose,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>estimator.predict_proba(X)</code></summary>

#### `estimator.predict_proba(X)`

- **역할:** sample별 class 확률 추정치를 반환합니다.
- **입력·반환:** feature 행렬을 받고 보통 `[samples, classes]` 확률 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** threshold·ROC·확률 기반 metric을 계산합니다.
- **주의점:** 모든 estimator가 지원하지 않으며 class 열 순서는 `classes_`를 따릅니다.
- **공식 문서:** [Estimator.predict_proba](https://scikit-learn.org/stable/glossary.html#term-predict-proba)

</details>

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
classifier = model.named_steps["classifier"]
probabilities = model.predict_proba(validation_x)

probability_table = pd.DataFrame(probabilities, columns=classifier.classes_)
probability_table.insert(0, "actual", validation_y.reset_index(drop=True))
probability_table.insert(1, "predicted", predictions)

assert np.allclose(probabilities.sum(axis=1), 1.0)
probability_table.head()

## 7. Accuracy만 보지 않는 평가

Macro F1은 각 클래스를 동일한 비중으로 평균내어 소수 클래스 성능 저하를 드러냅니다.
지원 건수(`support`)와 클래스별 precision/recall을 함께 봅니다.

## 8. Confusion matrix

행은 실제 레이블, 열은 예측 레이블입니다. 어떤 클래스 쌍이 반복해서 혼동되는지 확인합니다.

In [ ]:
report = evaluate_model(model, validation_x, validation_y)
print(
    {
        "accuracy": round(report.accuracy, 4),
        "macro_f1": round(report.macro_f1, 4),
        "weighted_f1": round(report.weighted_f1, 4),
    }
)

pd.DataFrame(
    {
        label: {
            "precision": metrics.precision,
            "recall": metrics.recall,
            "f1": metrics.f1,
            "support": metrics.support,
        }
        for label, metrics in report.per_class.items()
    }
).T

In [ ]:
# sklearn의 텍스트 보고서도 비교해 봅니다.
print(classification_report(validation_y, predictions, zero_division=0))

In [ ]:
confusion = pd.DataFrame(
    report.confusion_matrix,
    index=pd.Index(report.labels, name="actual"),
    columns=pd.Index(report.labels, name="predicted"),
)
confusion

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.asarray -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
if plt is None:
    print(confusion)
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(
        confusion_matrix=np.asarray(report.confusion_matrix),
        display_labels=report.labels,
    ).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
    ax.set_title("Group-split validation confusion matrix")
    plt.tight_layout()
    plt.show()

## 9. 오류 분석 표

틀린 샘플을 confidence가 높은 순으로 보면 모델이 확신하며 틀린 데이터·레이블 문제를 먼저 찾을 수 있습니다.
데이터가 작고 쉬우면 오류가 0개일 수 있으므로 low-margin 정답도 함께 관찰합니다.

## 10. Leakage 실험: 행 분할과 group 분할 비교

행 무작위 분할에서는 같은 case의 변형 문장이 양쪽에 들어갈 수 있습니다.
성능 차이가 항상 크게 나타나는 것은 아니지만, 평가 가정 자체가 깨졌다는 사실이 중요합니다.

In [ ]:
errors = build_error_table(model, split.validation, schema)

if errors.empty:
    margins = (
        np.sort(probabilities, axis=1)[:, -1] - np.sort(probabilities, axis=1)[:, -2]
    )
    uncertain = split.validation[["ticket_id", "case_id", "text", "label"]].copy()
    uncertain["predicted_label"] = predictions
    uncertain["confidence"] = probabilities.max(axis=1)
    uncertain["margin"] = margins
    print("오분류가 없어 margin이 작은 정답 사례를 표시합니다.")
    analysis_table = uncertain.sort_values("margin").head(10)
else:
    analysis_table = errors.head(10)

analysis_table

<!-- api-explanations:start -->
<!-- api-explanations:keys=train_test_split -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>train_test_split(*arrays, test_size=None, random_state=None, ...)</code></summary>

#### `train_test_split(*arrays, test_size=None, random_state=None, ...)`

- **역할:** 배열들을 같은 index 기준으로 train과 test로 나눕니다.
- **입력·반환:** 하나 이상의 배열을 받고 각 배열의 train·test 조각을 순서대로 반환합니다.
- **이 셀에서 쓰는 이유:** 평가 데이터 누수를 막고 동일 분할을 재현합니다.
- **주의점:** 분류에서는 `stratify`와 `random_state` 필요 여부를 확인해야 합니다.
- **공식 문서:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
row_train, row_validation = train_test_split(
    tickets,
    test_size=0.2,
    random_state=42,
    stratify=tickets["label"],
)
overlap = set(row_train["case_id"]) & set(row_validation["case_id"])

row_model = build_model(config, schema)
row_model.fit(row_train.loc[:, schema.model_feature_columns], row_train["label"])
row_report = evaluate_model(
    row_model,
    row_validation.loc[:, schema.model_feature_columns],
    row_validation["label"],
)

print("leaked case count:", len(overlap))

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "split": "random rows (leaky)",
            "accuracy": row_report.accuracy,
            "macro_f1": row_report.macro_f1,
            "overlapping_cases": len(overlap),
        },
        {
            "split": "stratified groups",
            "accuracy": report.accuracy,
            "macro_f1": report.macro_f1,
            "overlapping_cases": 0,
        },
    ]
).set_index("split")

assert comparison.loc["stratified groups", "overlapping_cases"] == 0
comparison

## 11. Feature leakage가 없는지 vocabulary 확인

TF-IDF vocabulary는 train에서만 학습되었습니다. validation 전용 토큰을 만들어 변환해도 vocabulary 크기는 변하지 않습니다.

## 12. Logistic Regression의 feature 가중치

양의 계수가 큰 feature는 해당 클래스로 log-odds를 밀어 올립니다.
상관관계 기반 설명이지 인과 설명은 아니며, 문자 n-gram은 조각 단위로 읽어야 합니다.

In [ ]:
text_vectorizer = preprocessor.named_transformers_["text_tfidf"]
vocabulary_size_before = len(text_vectorizer.vocabulary_)

_ = model.predict(
    pd.DataFrame(
        {
            "text": ["validationonlytoken_xyz"],
            "channel": ["unknown"],
            "priority": ["unknown"],
            "customer_tier": ["unknown"],
        }
    )
)

assert len(text_vectorizer.vocabulary_) == vocabulary_size_before
assert "validationonlytoken_xyz" not in text_vectorizer.vocabulary_
print("vocabulary size:", vocabulary_size_before)

In [ ]:
all_feature_names = preprocessor.get_feature_names_out()
coefficients = classifier.coef_

top_features: dict[str, list[tuple[str, float]]] = {}
for class_index, label in enumerate(classifier.classes_):
    top_indices = np.argsort(coefficients[class_index])[-8:][::-1]
    top_features[str(label)] = [
        (
            str(all_feature_names[index]),
            round(float(coefficients[class_index, index]), 4),
        )
        for index in top_indices
    ]

pd.DataFrame(top_features)

## 13. 새 문의 추론

학습과 동일한 Pipeline을 통과시키므로 정규화, TF-IDF, metadata 인코딩 순서를 수동으로 맞출 필요가 없습니다.

## 14. TODO: 사람 검토 threshold

운영에서는 confidence가 낮은 예측을 자동 처리하지 않고 사람에게 보낼 수 있습니다.
threshold는 비용과 대표 validation 데이터에 맞춰 조정해야 하며 0.7은 단지 실습값입니다.

In [ ]:
new_tickets = pd.DataFrame(
    [
        {
            "text": "카드 결제가 두 번 처리됐어요",
            "channel": "chat",
            "priority": "high",
            "customer_tier": "pro",
        },
        {
            "text": "로그인이 계속 실패하고 복구 코드도 안 와요",
            "channel": "web",
            "priority": "medium",
            "customer_tier": "free",
        },
        {
            "text": "배송 조회가 며칠째 같은 위치예요",
            "channel": "email",
            "priority": "low",
            "customer_tier": "business",
        },
    ]
)

inference = predict_tickets(model, new_tickets)
assert len(inference) == 3
inference

In [ ]:
def route_for_review(
    prediction_frame: pd.DataFrame, threshold: float = 0.7
) -> pd.DataFrame:
    # TODO 1: confidence가 threshold보다 낮으면 needs_review=True로 표시하세요.
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be in [0, 1]")
    output = prediction_frame.copy()
    output["needs_review"] = output["confidence"].lt(threshold)
    return output


routed = route_for_review(inference, threshold=0.7)
assert routed["needs_review"].dtype == bool
routed[["text", "predicted_label", "confidence", "needs_review"]]

## 직접 바꿔 볼 실험

- TODO 2: `ngram_range=(1, 3)`과 `(3, 6)`의 macro F1, feature 수, 학습 시간을 비교하세요.
- TODO 3: metadata 가중치를 0 또는 1로 바꾸고 channel 편향을 분석하세요.
- TODO 4: 가장 margin이 작은 문장의 일부 단어를 바꿔 확률 변화를 기록하세요.
- TODO 5: `C`를 0.1, 1, 10으로 바꾸고 과소/과적합 경향을 비교하세요.
- TODO 6: 오류 유형을 `ambiguous`, `label_issue`, `missing_context`로 직접 태깅하세요.

기준 모델이 확보되면 이후 PyTorch 모델은 단순 정확도뿐 아니라 비용·지연·유지보수 측면에서도 이 기준선과 비교해야 합니다.